# Data Reading

In [1]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
        .appName("Silver")

        # Delta Lake
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

        # Memory (CRITICAL for 100M+ rows)
        .config("spark.driver.memory", "8g")
        .config("spark.executor.memory", "8g")

        # CPU cores
        .config("spark.driver.cores", "4")

        # Shuffle tuning
        .config("spark.sql.shuffle.partitions", "200")

        # Adaptive query optimization
        .config("spark.sql.adaptive.enabled", "true")

        # Prevent huge driver result crashes
        .config("spark.driver.maxResultSize", "2g")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()

In [2]:
#importing modules
from pyspark.sql.functions import *
from pyspark.sql.types import *
import os
import logging

In [3]:
log_dir = "../logs"
os.makedirs(log_dir, exist_ok=True)

logging.basicConfig(
    filename=os.path.join(log_dir, "02_bronze_to_silver.log"),
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [4]:
logging.info(f"Processing Silver Layer")

In [5]:
df_trip_data = spark.read\
                        .format("delta") \
                        .load(r"..\01_storage_bronze\yellow_taxi")

In [6]:
df_trip_data.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+--------------------+----+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|bronze_ingestion_time|         source_file|year|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+--------------------+----+------

Surface Level Exploration

In [7]:
df_trip_data.count()

123897407

In [8]:
df_trip_data.describe().show()

+-------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+-------------------+------------------+------------------+---------------------+------------------+--------------------+------------------+--------------------+------------------+------------------+
|summary|          VendorID|   passenger_count|     trip_distance|        RatecodeID|store_and_fwd_flag|      PULocationID|     DOLocationID|      payment_type|       fare_amount|             extra|            mta_tax|        tip_amount|      tolls_amount|improvement_surcharge|      total_amount|congestion_surcharge|       airport_fee|         source_file|              year|cbd_congestion_fee|
+-------+------------------+------------------+------------------+------------------+------------------+------------------+-----------------+------------------+------------------+------------------+--------

In [9]:
df_trip_data.select([
    approx_count_distinct(c).alias(c)
    for c in df_trip_data.columns
]).show()


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+-----------+----+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|bronze_ingestion_time|source_file|year|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+---------------------+-----------+----+------------------+
|       4|   

# Data Transformations

Column Standardization

In [10]:
df_trip_data = (
    df_trip_data
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime")
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")
)

Null Handling

In [11]:
df_trip_data = (
    df_trip_data
    .withColumn("passenger_count",
        when(col("passenger_count").isNull(), 1)
        .otherwise(col("passenger_count"))
    )
    .withColumn("RatecodeID",
        when(col("RatecodeID").isNull(), 99)
        .otherwise(col("RatecodeID"))
    )
    .withColumn("store_and_fwd_flag",
        when(col("store_and_fwd_flag").isNull(), "N")
        .otherwise(col("store_and_fwd_flag"))
    )
    .withColumn("congestion_surcharge",
        when(col("congestion_surcharge").isNull(), 0.0)
        .otherwise(col("congestion_surcharge"))
    )
    .withColumn("airport_fee",
        when(col("airport_fee").isNull(), 0.0)
        .otherwise(col("airport_fee"))
    )
    .withColumn("cbd_congestion_fee",
        when(col("cbd_congestion_fee").isNull(), 0.0)
        .otherwise(col("cbd_congestion_fee"))
    )
)

Flag Coluumns for analytics

In [12]:
df_trip_data= (
    df_trip_data
    .withColumn("is_airport_trip",
        when(col("airport_fee") > 0, 1).otherwise(0)
    )
    .withColumn("is_cbd_trip",
        when(col("cbd_congestion_fee") > 0, 1).otherwise(0)
    )
    .withColumn("has_extra_charges",
        when(
            (col("extra") +
             col("mta_tax") +
             col("tolls_amount") +
             col("congestion_surcharge")) > 0,
            1
        ).otherwise(0)
    )
)

Business Logic

In [13]:
df_trip_data = df_trip_data.filter(
    (col("fare_amount").between(3, 200)) &
    (col("total_amount") > 0) &
    (col("trip_distance").between(0.5, 100)) &
    (col("passenger_count").between(1, 6)) &
    (col("dropoff_datetime") > col("pickup_datetime"))
)

Feature Engineering

In [14]:
df_trip_data = (

    df_trip_data

    .withColumn(
        "trip_date",
        to_date("dropoff_datetime")
    )

    .withColumn(
        "trip_year",
        year("dropoff_datetime")
    )

    .withColumn(
        "pickup_hour",
        hour("pickup_datetime")
    )

    .withColumn(
        "pickup_day",
        dayofweek("pickup_datetime")
    )

    .withColumn(
        "pickup_month",
        month("pickup_datetime")
    )

    .withColumn(
        "pickup_year",
        year("pickup_datetime")
    )
)

In [15]:
df_trip_data = df_trip_data.withColumn(
    "extra_charges",
    round(col("total_amount") - col("fare_amount"), 2)
)

In [16]:
df_trip_data = df_trip_data.withColumn(
    "trip_duration_seconds",
    col("dropoff_datetime").cast("long") -
    col("pickup_datetime").cast("long")
)

In [17]:
df_trip_data = (
    df_trip_data
    .filter(col("trip_duration_seconds") > 0)
    .withColumn(
        "trip_duration_minutes",
        round(col("trip_duration_seconds") / 60, 2)
    )
)

In [18]:
df_trip_data = df_trip_data.withColumn(
    "fare_per_mile",
    round(col("fare_amount") / col("trip_distance"), 2)
)

In [19]:
df_trip_data = df_trip_data.filter(
    (col("fare_per_mile") > 0) &
    (col("fare_per_mile") < 100)
)

Validation

In [20]:
df_trip_data.count()

110878479

In [21]:
df_trip_data.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- bronze_ingestion_time: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- cbd_congesti

In [22]:
df_trip_data.select("fare_per_mile").summary().show()

+-------+------------------+
|summary|     fare_per_mile|
+-------+------------------+
|  count|         110878479|
|   mean|  7.61990749241513|
| stddev|3.2335084073615166|
|    min|              0.03|
|    25%|              5.58|
|    50%|              7.13|
|    75%|              8.94|
|    max|             99.94|
+-------+------------------+



Keeping only required columns

In [23]:
df_trip_data = df_trip_data.drop("source_file")

In [24]:
df_trip_data = (
    df_trip_data
    .withColumnRenamed("VendorID", "vendor_id")
    .withColumnRenamed("PULocationID", "pu_location_id")
    .withColumnRenamed("DOLocationID", "do_location_id")
    .withColumnRenamed("RatecodeID", "ratecode_id")
)

Write

In [26]:

try:
    logging.info("Began Writing to silver layer")
    (
        df_trip_data
        .repartition(8, "trip_year")
        .write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .partitionBy("trip_year")
        .save(r"..\02_storage_silver\yellow_taxi")
    )
    logging.info("Successfully processed to silver layer")
except Exception as e:
    logging.exception(f"Error processing in silver layer: {e}")